In [1]:
import json
import torch
from pathlib import Path
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)
from trl import SFTConfig, SFTTrainer

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

CWD = Path.cwd()

if (CWD / "data").exists():
    BASE_DIR = CWD
else:
    BASE_DIR = CWD.parent

DATA_DIR = BASE_DIR / "data" / "processed"
MODEL_DIR = BASE_DIR / "models" / "qwen2.5-3b-medical-lora"

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.cuda.is_available())
print("Projeto:", BASE_DIR)

C:\Users\Diogo\anaconda3\envs\fiap_fase3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0912 20:14:14.250000 5152 site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


GPU: NVIDIA GeForce RTX 3060
CUDA: True
Projeto: C:\Users\Diogo\tech-challenge-FIAP-fase3-assistente-medico


In [2]:
def carregar_jsonl(caminho):
    dados = []

    with open(caminho, "r", encoding="utf-8") as f:
        for linha in f:
            dados.append(json.loads(linha))

    return dados

train_data = carregar_jsonl(DATA_DIR / "train.jsonl")
validation_data = carregar_jsonl(DATA_DIR / "validation.jsonl")

train_dataset = Dataset.from_list(train_data)
validation_dataset = Dataset.from_list(validation_data)

print("Treino:", len(train_dataset))
print("Validação:", len(validation_dataset))
print("Colunas:", train_dataset.column_names)

Treino: 40
Validação: 20
Colunas: ['categoria', 'protocolo', 'messages']


In [3]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    dtype=torch.bfloat16
)

model.config.use_cache = False

print("Modelo carregado.")
print("Dispositivo:", model.device)

Loading weights: 100%|██████████| 434/434 [00:03<00:00, 123.99it/s]


Modelo carregado.
Dispositivo: cuda:0


In [4]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


In [5]:
training_args = SFTConfig(
    output_dir=str(MODEL_DIR),

    num_train_epochs=3,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=4,

    learning_rate=1e-4,

    fp16=False,
    bf16=True,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    },

    max_length=512,

    logging_strategy="steps",
    logging_steps=2,

    eval_strategy="epoch",
    save_strategy="epoch",

    save_total_limit=2,

    optim="adamw_torch",

    report_to="none",

    seed=42
)

print("Configuração criada.")

Configuração criada.


In [6]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer
)

print("Trainer criado.")

Truncating train dataset: 100%|██████████| 40/40 [00:00<00:00, 4859.16 examples/s]
Dropping fully masked examples from train dataset: 100%|██████████| 40/40 [00:00<?, ? examples/s]
Dropping fully masked examples from eval dataset: 100%|██████████| 20/20 [00:00<?, ? examples/s]

Trainer criado.


In [7]:
print(
    "VRAM alocada antes do treinamento:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

print(
    "VRAM reservada antes do treinamento:",
    round(torch.cuda.memory_reserved() / 1024**3, 2),
    "GB"
)

VRAM alocada antes do treinamento: 2.56 GB
VRAM reservada antes do treinamento: 3.26 GB


In [8]:
resultado_treino = trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,1.225413,1.034851,1.170074,9124.000000,0.802590
2,0.504171,0.484138,0.444148,18248.000000,0.898021
3,0.400266,0.432539,0.432654,27372.000000,0.900814


In [9]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

print("Modelo LoRA salvo em:")
print(MODEL_DIR)

Modelo LoRA salvo em:
C:\Users\Diogo\tech-challenge-FIAP-fase3-assistente-medico\models\qwen2.5-3b-medical-lora


In [10]:
import os

print(os.listdir(MODEL_DIR))

['adapter_config.json', 'adapter_model.safetensors', 'chat_template.jinja', 'checkpoint-20', 'checkpoint-30', 'README.md', 'tokenizer.json', 'tokenizer_config.json']
